# 01. 토크나이저 — 토큰이 곧 비용이다

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/02_tokenizer.ipynb)

> 🧭 **스토리** — 한빛테크 신입 AI 담당자인 당신에게 첫 업무가 떨어졌다. "사내 규정집을 AI에게 읽히면 돈이 얼마나 드나?" 견적을 내려면 먼저 **토큰**을 세어야 한다.
>
> 📖 **이론 연결** — 이론1 p70~85 (BPE·어휘 크기·한국어 토큰 페널티·숫자/공백·NFC·토큰 세기 실무) · ⏱ **11분** · 🖥 **CPU**
>
> GPU가 필요 없습니다. 기본 런타임(CPU)으로 실행하세요.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
토크나이저 3종(tiktoken + 허깅페이스 2종)과 OpenAI 클라이언트만 설치합니다. 모델 본체는 받지 않습니다(토크나이저 파일은 수 MB).

In [ ]:
%pip -q install tiktoken "transformers>=4.54" openai

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

## 1. 토큰은 글자가 아니다 🔵
OpenAI 모델이 쓰는 `o200k_base` 인코딩으로 한국어와 영어 문장을 쪼개 봅니다. **토큰 ≠ 글자 ≠ 단어**라는 것을 눈으로 확인하세요.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # GPT-4o·GPT-5·GPT-6 계열이 쓰는 인코딩 (어휘 약 20만 개)

def show_tokens(text, enc=enc):
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]          # 토큰 하나씩 되돌려 '조각'을 봅니다
    print(f"{text!r}\n  글자 {len(text)}개 → 토큰 {len(ids)}개 : {pieces}")
    return ids

show_tokens("안녕하세요")
show_tokens("Hello")
show_tokens("출장비는 복귀 후 7일 이내에 정산한다.")
show_tokens("Travel expenses must be settled within 7 days after returning.")

💡 **해석** — 한국어는 글자 수보다 토큰 수가 더 많거나 비슷하고(한 글자가 2~3토큰으로 쪼개지기도 함), 영어는 단어 하나가 토큰 하나인 경우가 많습니다. 토큰 조각에 `�`처럼 깨진 글자가 보이면 **바이트 수준 BPE**가 한 글자를 바이트 단위로 쪼갠 것입니다(이론1 p73).

## 2. 한국어 토큰 페널티 — 같은 뜻, 몇 배? 🔵
규정집 문장 10개를 한·영으로 번역해 둔 병렬 데이터로 **같은 의미를 전달하는 데 드는 토큰 수**를 비교합니다.

In [ ]:
import pandas as pd
pairs = load_json("parallel_ko_en.json")
rows = []
for p in pairs:
    ko, en = len(enc.encode(p["ko"])), len(enc.encode(p["en"]))
    rows.append({"ko": p["ko"][:22] + "…", "ko_tokens": ko, "en_tokens": en, "ratio": round(ko / en, 2)})
df = pd.DataFrame(rows)
display(df)
print(f"평균 페널티(한국어 토큰 ÷ 영어 토큰): {df['ratio'].mean():.2f}배  (최대 {df['ratio'].max():.2f}배)")

💡 **해석** — 같은 뜻을 한국어로 보내면 영어보다 토큰을 더 씁니다. 고려대 연구는 cl100k·Llama·EXAONE 토크나이저에서 **최대 3배**를 보고했고(이론1 p76), 이 표의 평균이 그보다 작다면 `o200k_base`가 한국어 어휘를 더 많이 담고 있기 때문입니다. **비용·지연·컨텍스트 소모가 그만큼 커진다**는 사실은 그대로입니다.

## 3. 토크나이저가 다르면 토큰 수도 다르다 🔵
같은 문장을 네 가지 토크나이저로 세어 봅니다. 오늘 오후에 쓸 모델들의 토크나이저입니다.
- `o200k_base` (OpenAI GPT-6 계열) · `EXAONE-4.0` (LG, 한국어 특화) · `Qwen3` (알리바바) · `multilingual-e5` (임베딩 모델, XLM-R 토크나이저)

In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

hf_names = {"EXAONE-4.0": "LGAI-EXAONE/EXAONE-4.0-1.2B",
            "Qwen3": "Qwen/Qwen3-1.7B",
            "e5(XLM-R)": "intfloat/multilingual-e5-small"}
hf_toks = {k: AutoTokenizer.from_pretrained(v) for k, v in hf_names.items()}   # 토크나이저 파일만 다운로드(수 MB)

sample_ko = " ".join(p["ko"] for p in pairs)   # 한국어 10문장
sample_en = " ".join(p["en"] for p in pairs)   # 영어 10문장
counts = {"o200k_base": (len(enc.encode(sample_ko)), len(enc.encode(sample_en)))}
for k, t in hf_toks.items():
    counts[k] = (len(t.encode(sample_ko, add_special_tokens=False)), len(t.encode(sample_en, add_special_tokens=False)))

cmp = pd.DataFrame(counts, index=["한국어 10문장", "영어 10문장"]).T
cmp["어휘 크기"] = [enc.n_vocab] + [len(t) for t in hf_toks.values()]
display(cmp)
cmp[["한국어 10문장", "영어 10문장"]].plot.bar(figsize=(7, 3.5), title="같은 문장, 토크나이저별 토큰 수")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

💡 **해석** — 어휘가 크다고 항상 토큰이 적게 나오지는 않습니다. **어떤 언어의 조각을 어휘에 많이 넣었느냐**가 결정합니다. 강사 환경 측정에서는 한국어 10문장이 e5(XLM-R, 25만 어휘·100개 언어) 151 < EXAONE-4.0(10만 어휘, 한국어 특화) 158 < o200k_base 167 < Qwen3 196 토큰이었습니다. EXAONE은 어휘가 가장 작은데도 한국어에서 2위입니다. 반대로 영어 문장은 네 토크나이저가 거의 같습니다. "한국어 서비스면 한국어 토큰 효율이 좋은 모델이 싸다"는 실무 판단이 여기서 나옵니다(이론1 p77).

## 4. 숫자·공백·유니코드 — 토큰 경계의 함정 🔵
LLM이 산수를 틀리고, 글자 수를 못 세고, 눈에 똑같은 문장을 다르게 처리하는 이유의 상당 부분이 **토큰 경계**에 있습니다.

In [ ]:
import unicodedata
print("--- 숫자: 몇 자리씩 끊길까 ---")
for s in ["1234567", "1,234,567", "2026년 1월 1일", "310,000원"]:
    show_tokens(s)

print("\n--- 공백: 앞에 띄어쓰기 하나가 다른 토큰 ---")
for s in ["hello", " hello", "hello ", "출장", " 출장"]:
    show_tokens(s)

print("\n--- 유니코드 정규화: 눈에 같은 '한'이 다른 바이트 ---")
nfc = unicodedata.normalize("NFC", "한빛테크 출장")      # 완성형(일반적인 저장 형태)
nfd = unicodedata.normalize("NFD", "한빛테크 출장")      # 자모 분리형(macOS 파일명·일부 시스템 export)
print("NFC 길이", len(nfc), "토큰", len(enc.encode(nfc)), "|", [hex(ord(c)) for c in nfc[:1]])
print("NFD 길이", len(nfd), "토큰", len(enc.encode(nfd)), "|", [hex(ord(c)) for c in nfd[:3]])
print("같은 문자열인가?", nfc == nfd, "→ 정규화 후:", unicodedata.normalize("NFC", nfd) == nfc)

💡 **해석**
- 숫자는 1~3자리 덩어리로 끊깁니다. `1234567`과 `1,234,567`의 토큰 경계가 다르므로 모델에게 "같은 수"가 아닙니다(이론1 p80). 금액이 많은 문서는 쉼표 표기를 통일하세요.
- ` hello`(앞 공백)와 `hello`는 다른 토큰입니다. 프롬프트 끝에 공백을 붙이면 다음 토큰 예측이 흔들립니다(이론1 p83).
- NFD 문자열은 글자 수도 토큰 수도 늘어나고, `==` 비교와 검색이 실패합니다. **입력 전처리 1번은 `unicodedata.normalize("NFC", s)`** 입니다(이론1 p86). 노트북 06에서 실제 파일로 다시 봅니다.

## 5. 규정집 견적 — 토큰 × 단가 🔵
오늘 하루 다룰 「한빛테크 출장·복리후생 규정」 전체(마크다운)를 한 번 보내면 얼마일까요? 2026-10 공식 단가로 계산합니다.

In [ ]:
# ⚙️ 비용 계산 — 2026-10 기준 공식 단가(USD per 1M tokens). https://developers.openai.com/api/docs/pricing
PRICES = {  # (입력, 캐시 입력, 출력)
    "gpt-6-luna":   (0.10, 0.01, 0.50),
    "gpt-5-nano":   (0.05, 0.005, 0.40),
    "gpt-5.6-luna": (0.20, 0.02, 1.20),
    "gpt-6-astra":  (10.0, 1.0, 50.0),
    "text-embedding-3-small": (0.02, 0.02, 0.0),
}
def cost_usd(model: str, in_tok: int, out_tok: int = 0, cached_tok: int = 0) -> float:
    p_in, p_cache, p_out = PRICES.get(model, (0.10, 0.01, 0.50))
    return ((in_tok - cached_tok) * p_in + cached_tok * p_cache + out_tok * p_out) / 1_000_000
def cost_krw(usd: float, rate: float = 1400.0) -> float:
    return usd * rate

In [ ]:
rules_md = load_text("hanbit_rules_gt.md")
n_tok = len(enc.encode(rules_md))
print(f"규정집: 글자 {len(rules_md):,}자 · 토큰 {n_tok:,}개 (o200k_base 기준)")
print()
print(f"{'모델':<14}{'1회 입력 비용(USD)':>20}{'1,000회(USD)':>14}{'1,000회(원)':>12}")
for m in ["gpt-6-luna", "gpt-5-nano", "gpt-5.6-luna", "gpt-6-astra"]:
    c = cost_usd(m, n_tok)
    print(f"{m:<14}{c:>20.5f}{c*1000:>14.2f}{cost_krw(c*1000):>12,.0f}")
print("\n캐시 입력(같은 접두부 반복)이면:", f"{cost_usd('gpt-6-luna', n_tok, cached_tok=n_tok):.5f} USD  ← 10분의 1")

💡 **해석** — 규정집 전체를 매 질문마다 통째로 보내도 소형 모델이면 1,000번에 몇백 원 수준입니다. 그런데 문서가 100배 커지면? 플래그십 모델이면? 표의 숫자를 100배 해 보세요. **"필요한 부분만 골라 보내기"(RAG)와 "캐시"가 왜 돈이 되는지**가 여기서 나옵니다(이론1 p17·p42·p44).

## 6. 추정 vs 실제 — API `usage` 로 확인 🔵
tiktoken은 **추정**입니다. 실제 과금은 서버가 세어 돌려주는 `usage` 값입니다. 둘을 비교합니다.
(GPT-6 계열은 아직 `tiktoken.encoding_for_model`에 없어서 `o200k_base`로 근사합니다 — 이론1 p16)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
from openai import OpenAI
client = OpenAI()

try:
    tiktoken.encoding_for_model(MODEL_API)
except KeyError as e:
    print("encoding_for_model 실패 →", e, "\n→ get_encoding('o200k_base')로 근사합니다.")

prompt = pairs[0]["ko"] + " " + pairs[1]["ko"]
est = len(enc.encode(prompt))
try:
    r = client.responses.create(model=MODEL_API, input=prompt, instructions="한 문장으로 답하세요.",
                                reasoning={"effort": "none"}, max_output_tokens=60)
    u = r.usage
    print("답변:", r.output_text)
    print(f"입력 토큰 — tiktoken 추정 {est} vs 서버 실제 {u.input_tokens} (지시문·역할 토큰이 더해져 실제가 큽니다)")
    print(f"출력 토큰 {u.output_tokens} · 비용 {cost_usd(MODEL_API, u.input_tokens, u.output_tokens):.6f} USD")
except Exception as e:
    print("API 호출 실패(키·네트워크 확인):", type(e).__name__, getattr(e, "status_code", ""))  # 오류 본문은 키 일부를 포함할 수 있어 출력하지 않습니다

💡 **해석** — 실제 입력 토큰이 추정보다 큽니다. 메시지 역할 구분자, 시스템 지시, 모델별 특수 토큰이 더해지기 때문입니다. **비용 추정은 tiktoken으로, 정산은 `usage`로** 하세요. 로그에 `usage`를 남기는 습관이 비용 관리의 출발점입니다.

## 7. 심화 🟠 (시간 남을 때)
1. `my_text`에 **본인이 자주 쓸 문장**(업무 메일·보고서 한 단락)을 넣고 토큰 수·비용을 구해 보세요.
2. 인코딩을 `cl100k_base`(GPT-4 세대)로 바꾸면 한국어 토큰이 몇 % 늘어나는지 확인하세요.

In [ ]:
my_text = "다음 주 화요일 부산 출장 건으로 숙박비 상한과 일비 지급 기준을 확인 부탁드립니다."   # ← 바꿔 보세요
old = tiktoken.get_encoding("cl100k_base")
for name, e in [("o200k_base", enc), ("cl100k_base", old)]:
    n = len(e.encode(my_text))
    print(f"{name:<12} 토큰 {n:>3}개 · gpt-6-luna 입력 기준 {cost_usd('gpt-6-luna', n)*1e6:.1f} 마이크로달러")
print(f"구세대 인코딩 대비 절감: {100*(1-len(enc.encode(my_text))/len(old.encode(my_text))):.0f}%")

### 🤔 생각해보기

한국어 서비스에서 **토큰 비용을 줄이는 방법**을 세 가지 이상 적어 보세요. 모델을 바꾸는 것 말고도 있을까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **캐시**: 변하지 않는 지시문·문서를 앞에 두어 캐시 입력 단가(10분의 1)를 받는다.
- **필요한 부분만**: 문서 전체 대신 검색으로 고른 조각만 보낸다(RAG, 노트북 11).
- **출력 상한**: `max_output_tokens`로 긴 답을 막는다. 출력 단가는 입력의 5배다.
- **한국어 효율이 좋은 토크나이저의 모델**: 같은 문장을 더 적은 토큰으로 쓴다(EXAONE 등).
- **전처리**: 중복 공백·불필요한 서식·반복 문구 제거, NFC 정규화로 쓸데없는 토큰을 없앤다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 토큰은 모델의 '글자'이고 비용·컨텍스트·한계의 단위다.
- 한국어는 영어보다 토큰을 더 쓴다. 토크나이저에 따라 차이가 크다.
- 숫자·공백·유니코드 정규화가 토큰 경계를 바꾼다. 전처리 1번은 NFC.
- 추정은 tiktoken, 정산은 `usage`. GPT-6은 `o200k_base`로 근사.

> 다음 노트북: **03_open_llm_modelcard — 모델 카드 코드로 오픈 LLM 띄우기**